# 🛠️ **Testing Tool Providers Dynamically**

This notebook will:

- List all registered tools from the database, displaying their names and GUIDs explicitly.
- Dynamically load the corresponding tool scripts from the `extensions` directory.
- Instantiate each tool using the shared `ToolProviderBase` infrastructure.
- Test tool invocation explicitly for both success and failure cases.
- Verify the tool invocation logging clearly.


### Configuring Project Path

This cell ensures the notebook can locate the project's modules by adding the project root directory to Python’s import path (`sys.path`). This setup is required for relative imports from the main application (`app`) to work correctly within the notebook environment.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

c:\Repos\codecritic


In [2]:
from sqlalchemy import create_engine
from app.db.base import Base
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]
DB_PATH = PROJECT_ROOT / "experiments" / "codecritic.sqlite3"

engine = create_engine(f"sqlite:///{DB_PATH}")

# Explicitly drop and recreate your database schema
Base.metadata.drop_all(bind=engine)
Base.metadata.create_all(bind=engine)

print("✅ Database schema reset and recreated successfully.")

✅ Database schema reset and recreated successfully.


In [3]:
import sqlite3
from app.utilities.db import DB_PATH

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

schema = cur.execute("PRAGMA table_info(tool_invocation_log)").fetchall()
print(schema)

conn.close()


[(0, 'id', 'INTEGER', 1, None, 1), (1, 'experiment_id', 'VARCHAR', 1, None, 0), (2, 'round', 'INTEGER', 1, None, 0), (3, 'tool_provider_name', 'VARCHAR', 1, None, 0), (4, 'tool_provider_guid', 'VARCHAR', 1, None, 0), (5, 'invocation_parameters', 'VARCHAR', 1, None, 0), (6, 'stdout', 'VARCHAR', 0, None, 0), (7, 'stderr', 'VARCHAR', 0, None, 0), (8, 'return_code', 'INTEGER', 1, None, 0), (9, 'success', 'BOOLEAN', 1, None, 0), (10, 'error_message', 'VARCHAR', 0, None, 0), (11, 'timestamp', 'DATETIME', 1, None, 0)]


## 🔧 Database Initialization

In this step, we establish a connection to the SQLite database and explicitly initialize it, creating tables as defined in our SQLAlchemy models.

This step ensures that the database schema matches our SQLAlchemy definitions.

In [4]:
from sqlalchemy import select
from sqlalchemy.orm import Session
from app.db.models import ToolConfig
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_tools import seed_tools

with Session(bind=engine) as session:
    seed_prompts(session)
    seed_tools(session)
    tools = session.execute(select(ToolConfig)).scalars().all()

print("✅ Data seeded successfully.")

print("Registered tools:")
for tool in tools:
    print(f"- {tool.name}: {tool.guid}")
print(PROJECT_ROOT)


Seeded AgentPrompt GUID: 269b0b69-da58-436b-989d-524eb23e1358
Seeded SystemPrompt GUID: 13b2eb52-7453-4a40-b7b7-fd83ec48b62f
Seeded tool configurations successfully.
✅ Data seeded successfully.
Registered tools:
- Black Formatter: 729ecdd2-c379-45b8-bdb9-b5dcc26e5671
- SonarCloud Analyzer: 2210dfc4-f81e-42c8-9dcb-19537c927ddc
- Ruff Linter: ab6a38c0-af3a-4c35-8bf4-a4c5bae8d07e
- Radon Analyzer: a3c33a86-f811-442d-888a-4d981c00fff4
- Mypy Type Checker: d32c90d0-cb26-4880-89a0-5452db924ee7
- Docformatter Formatter: e36c0d78-703e-4f5d-ad1e-b52205e6bafe
c:\Repos\codecritic


## 📦 Dynamic Tool Loader

We dynamically load tool scripts explicitly by GUID from the extensions folder, instantiate them using the common `ToolProviderBase` class, and test invocation.


In [5]:
import sys
import importlib
from pathlib import Path
from sqlalchemy.orm import Session
from sqlalchemy import select, create_engine
from app.abstract_classes.tool_provider_base import ToolProviderBase
from app.db.models import ToolConfig
from app.factories.logging_provider import LoggingProvider

PROJECT_ROOT = Path.cwd().parents[2]
DB_PATH = PROJECT_ROOT / "experiments" / "codecritic.sqlite3"
engine = create_engine(f"sqlite:///{DB_PATH}")
logger = LoggingProvider()

test_file = PROJECT_ROOT / "tests" / "example.py"
test_file.parent.mkdir(parents=True, exist_ok=True)
test_file.write_text("print('hello world')\n", encoding="utf-8")

with Session(bind=engine) as session:
    tools = session.execute(select(ToolConfig)).scalars().all()

print("🔍 Starting explicit tool validation:\n")

for tool_config in tools:
    print(f"🧪 Tool: {tool_config.name}")

    try:
        tool_script_path = Path(tool_config.artifact_path)
        spec = importlib.util.spec_from_file_location(tool_config.guid, tool_script_path)
        tool_module = importlib.util.module_from_spec(spec)
        sys.modules[tool_config.guid] = tool_module
        spec.loader.exec_module(tool_module)

        # Explicitly find concrete ToolProvider subclass (excluding abstract base)
        tool_class = next(
            cls for cls in vars(tool_module).values()
            if isinstance(cls, type)
            and issubclass(cls, ToolProviderBase)
            and cls is not ToolProviderBase
        )

        tool_provider = tool_class(tool_config, logger=logger)
        result = tool_provider.run(str(test_file), experiment_id="test_run", round=1)

        print(f"✅ {tool_config.name} executed successfully.")
        print(f"Return code: {result.returncode}")
        print(f"Stdout:\n{result.stdout.strip()}")
        print(f"Stderr:\n{result.stderr.strip()}\n")

    except Exception as e:
        print(f"❌ {tool_config.name} failed explicitly:")
        print(e, "\n")


🔍 Starting explicit tool validation:

🧪 Tool: Black Formatter
✅ Black Formatter executed successfully.
Return code: 0
Stdout:

Stderr:


🧪 Tool: SonarCloud Analyzer
✅ SonarCloud Analyzer executed successfully.
Return code: 0
Stdout:
{}
Stderr:


🧪 Tool: Ruff Linter
✅ Ruff Linter executed successfully.
Return code: 0
Stdout:
All checks passed!
Stderr:


🧪 Tool: Radon Analyzer
✅ Radon Analyzer executed successfully.
Return code: 0
Stdout:

Stderr:


🧪 Tool: Mypy Type Checker
✅ Mypy Type Checker executed successfully.
Return code: 0
Stdout:
Success: no issues found in 1 source file
Stderr:


🧪 Tool: Docformatter Formatter
✅ Docformatter Formatter executed successfully.
Return code: 0
Stdout:

Stderr:




## 🧪 Testing Tool Invocations

We explicitly test both successful and intentional failure cases for each tool to verify correct invocation and logging.


In [9]:
import sys
import importlib
from pathlib import Path
from sqlalchemy.orm import Session
from sqlalchemy import select, create_engine
from app.abstract_classes.tool_provider_base import ToolProviderBase
from app.db.models import ToolConfig
from app.factories.logging_provider import LoggingProvider

PROJECT_ROOT = Path.cwd().parents[2]
DB_PATH = PROJECT_ROOT / "experiments" / "codecritic.sqlite3"
engine = create_engine(f"sqlite:///{DB_PATH}")
logger = LoggingProvider()

# Create clearly valid and invalid test files explicitly
valid_test_file = PROJECT_ROOT / "tests" / "example.py"
valid_test_file.parent.mkdir(parents=True, exist_ok=True)
valid_test_file.write_text("print('hello world')\n", encoding="utf-8")

invalid_test_file = PROJECT_ROOT / "tests" / "nonexistent_file.py"  # intentionally nonexistent

with Session(bind=engine) as session:
    tools = session.execute(select(ToolConfig)).scalars().all()

def load_tool_provider(tool_config: ToolConfig, logger: LoggingProvider) -> ToolProviderBase:
    tool_script_path = Path(tool_config.artifact_path)
    spec = importlib.util.spec_from_file_location(tool_config.guid, tool_script_path)
    tool_module = importlib.util.module_from_spec(spec)
    sys.modules[tool_config.guid] = tool_module
    spec.loader.exec_module(tool_module)

    tool_class = next(
        cls for cls in vars(tool_module).values()
        if isinstance(cls, type)
        and issubclass(cls, ToolProviderBase)
        and cls is not ToolProviderBase
    )

    return tool_class(tool_config, logger=logger)

print("🧪 Final Corrected Tool Invocations\n")

for tool_config in tools:
    print(f"\n🛠️ Tool: {tool_config.name} (GUID: {tool_config.guid})")

    provider = load_tool_provider(tool_config, logger)

    # Explicit VALID TEST (now using the correct file!)
    try:
        print("✅ Valid invocation test:")
        result = provider.run(str(valid_test_file), experiment_id="valid_test", round=1)
        print("Return code:", result.returncode)
        print("STDOUT:", result.stdout.strip())
        print("STDERR:", result.stderr.strip())
    except Exception as e:
        print("❌ Unexpected error during valid case:", e)

    # Explicit INVALID TEST (expected to fail, clearly using invalid file)
    try:
        print("❌ Invalid invocation test (expected failure):")
        result = provider.run(str(invalid_test_file), experiment_id="invalid_test", round=1)
        print("Return code:", result.returncode)
        print("STDOUT:", result.stdout.strip())
        print("STDERR:", result.stderr.strip())
    except Exception as e:
        print("✅ Caught expected failure:", e)


🧪 Final Corrected Tool Invocations


🛠️ Tool: Black Formatter (GUID: 729ecdd2-c379-45b8-bdb9-b5dcc26e5671)
✅ Valid invocation test:
Return code: 0
STDOUT: 
STDERR: 
❌ Invalid invocation test (expected failure):


Usage: python -m black [OPTIONS] SRC ...
Try 'python -m black -h' for help.

Error: Invalid value for 'SRC ...': Path 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py' does not exist.

Critical tool run error logged: black failed: Usage: python -m black [OPTIONS] SRC ...
Try 'python -m black -h' for help.

Error: Invalid value for 'SRC ...': Path 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py' does not exist.

Critical tool run error logged: ruff failed: 


✅ Caught expected failure: black failed: Usage: python -m black [OPTIONS] SRC ...
Try 'python -m black -h' for help.

Error: Invalid value for 'SRC ...': Path 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py' does not exist.


🛠️ Tool: SonarCloud Analyzer (GUID: 2210dfc4-f81e-42c8-9dcb-19537c927ddc)
✅ Valid invocation test:
Return code: 0
STDOUT: {}
STDERR: 
❌ Invalid invocation test (expected failure):
Return code: 0
STDOUT: {}
STDERR: 

🛠️ Tool: Ruff Linter (GUID: ab6a38c0-af3a-4c35-8bf4-a4c5bae8d07e)
✅ Valid invocation test:
Return code: 0
STDOUT: All checks passed!
STDERR: 
❌ Invalid invocation test (expected failure):
✅ Caught expected failure: ruff failed: 

🛠️ Tool: Radon Analyzer (GUID: a3c33a86-f811-442d-888a-4d981c00fff4)
✅ Valid invocation test:
Return code: 0
STDOUT: 
STDERR: 
❌ Invalid invocation test (expected failure):
Return code: 0
STDOUT: 
STDERR: 

🛠️ Tool: Mypy Type Checker (GUID: d32c90d0-cb26-4880-89a0-5452db924ee7)
✅ Valid invocation test:


mypy: can't read file 'c:\Repos\codecritic\tests\nonexistent_file.py': No such file or directory



Return code: 0
STDOUT: Success: no issues found in 1 source file
STDERR: 
❌ Invalid invocation test (expected failure):


Critical tool run error logged: mypy execution error: mypy: can't read file 'c:\Repos\codecritic\tests\nonexistent_file.py': No such file or directory



✅ Caught expected failure: mypy execution error: mypy: can't read file 'c:\Repos\codecritic\tests\nonexistent_file.py': No such file or directory


🛠️ Tool: Docformatter Formatter (GUID: e36c0d78-703e-4f5d-ad1e-b52205e6bafe)
✅ Valid invocation test:
Return code: 0
STDOUT: 
STDERR: 
❌ Invalid invocation test (expected failure):


[Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'

Critical tool run error logged: docformatter failed: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'



✅ Caught expected failure: docformatter failed: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'



## 📜 Verifying Logs Explicitly

Explicitly query the logs to confirm the tool invocation records were correctly created.


In [10]:
import sqlite3
import json

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

print("✅ Tool Invocation Logs:")
for row in cur.execute("SELECT experiment_id, round, tool_provider_name, tool_provider_guid, success, error_message FROM tool_invocation_log"):
    print(row)

conn.close()


✅ Tool Invocation Logs:
('test_run', 1, 'BlackToolProvider', '729ecdd2-c379-45b8-bdb9-b5dcc26e5671', 1, None)
('test_run', 1, 'SonarCloudToolProvider', '2210dfc4-f81e-42c8-9dcb-19537c927ddc', 1, None)
('test_run', 1, 'RuffToolProvider', 'ab6a38c0-af3a-4c35-8bf4-a4c5bae8d07e', 1, None)
('test_run', 1, 'RadonToolProvider', 'a3c33a86-f811-442d-888a-4d981c00fff4', 1, None)
('test_run', 1, 'MypyToolProvider', 'd32c90d0-cb26-4880-89a0-5452db924ee7', 1, None)
('test_run', 1, 'DocFormatterToolProvider', 'e36c0d78-703e-4f5d-ad1e-b52205e6bafe', 1, None)
('valid_test', 1, 'BlackToolProvider', '729ecdd2-c379-45b8-bdb9-b5dcc26e5671', 1, None)
('invalid_test', 1, 'BlackToolProvider', '729ecdd2-c379-45b8-bdb9-b5dcc26e5671', 0, "black failed: Usage: python -m black [OPTIONS] SRC ...\nTry 'python -m black -h' for help.\n\nError: Invalid value for 'SRC ...': Path 'c:\\\\Repos\\\\codecritic\\\\tests\\\\nonexistent_file.py' does not exist.\n")
('valid_test', 1, 'SonarCloudToolProvider', '2210dfc4-f81e-42c